In [29]:
from surprise import Dataset, Reader, SVD, NormalPredictor, accuracy
from surprise.model_selection import cross_validate, train_test_split

# dataset manipulation
import pandas as pd
import numpy as np

In [7]:
# load ratings
dataset_path = "../Transformed_Data/ratings_small.csv"
ratings_df = pd.read_csv(dataset_path)
ratings_df.head()

,UserID,MovieID,Rating
0,1,1193,5
1,1,661,3
2,1,914,3
3,1,3408,4
4,1,2355,5


In [9]:
# A reader is needed and only the rating_scale param is required.
reader = Reader(rating_scale=(1, 5))

In [12]:
ratings = Dataset.load_from_df(ratings_df[["UserID", "MovieID", "Rating"]], reader)

#### 1. First tests on ratings_small ratings dataset

In [15]:
cross_validate(NormalPredictor(), ratings, cv=2)

{'test_rmse': array([1.49949334, 1.49754944]),
 'test_mae': array([1.20198072, 1.19835263]),
 'fit_time': (0.060602664947509766, 0.06393218040466309),
 'test_time': (0.29247617721557617, 0.1667487621307373)}

In [16]:
# Run 5-fold cross-validation and print results
algo =  SVD()
cross_validate(algo, ratings, measures=["RMSE", "MAE"], cv=5, verbose=True)

Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.9258  0.9297  0.9225  0.9314  0.9323  0.9284  0.0037  
MAE (testset)     0.7339  0.7363  0.7312  0.7333  0.7346  0.7339  0.0017  
Fit time          1.09    1.05    1.05    1.04    1.07    1.06    0.02    
Test time         0.12    0.11    0.20    0.11    0.11    0.13    0.04    


{'test_rmse': array([0.92576436, 0.92974134, 0.92254079, 0.93142406, 0.93229293]),
 'test_mae': array([0.73385722, 0.73633263, 0.73118458, 0.73330493, 0.73463522]),
 'fit_time': (1.0891764163970947,
  1.0525140762329102,
  1.0477373600006104,
  1.0434396266937256,
  1.06520676612854),
 'test_time': (0.11646270751953125,
  0.11260604858398438,
  0.20024323463439941,
  0.11143088340759277,
  0.10933208465576172)}

In [ ]:
trainset, testset = train_test_split(ratings, test_size=0.25)

# We'll still use the famous SVD algorithm.
algo = SVD()

# Train the algorithm on the trainset, and predict ratings for the testset
algo.fit(trainset)
predictions = algo.test(testset)

# Then compute RMSE
accuracy.rmse(predictions)

RMSE: 0.9197


np.float64(0.9197261873669083)

#### 2. Comparison with custom collaborative filtering RS

In [23]:
trainset = ratings.build_full_trainset()

In [24]:
algo.fit(trainset)

In [25]:
uid = str(196)  # raw user id (as in the ratings file). They are **strings**!
iid = str(302)  # raw item id (as in the ratings file). They are **strings**!

# get a prediction for specific users and items.
pred = algo.predict(uid, iid, verbose=True)

user: 196        item: 302        r_ui = None   est = 3.60   {'was_impossible': False}


In [27]:
display(ratings_df.head())

,UserID,MovieID,Rating
0,1,1193,5
1,1,661,3
2,1,914,3
3,1,3408,4
4,1,2355,5


In [28]:
pred = algo.predict(1, 2355, verbose=True)

user: 1          item: 2355       r_ui = None   est = 4.28   {'was_impossible': False}


In [30]:
# loading custom collaborative prediction
custom_prediction = np.load("../transformed_Data/user_prediction.npy")

In [31]:
# Loading user indices correspondance
user_corres = np.load("../transformed_Data/user_corres_matrix_index.npy")
# Loading movie indices correspondance
movie_corres = np.load("../transformed_Data/movie_corres_matrix_index.npy")
# loading correspondance between movie indices and movie titles
movies_titles = pd.read_csv("../transformed_Data/MovieID_title.csv")

In [32]:
user_corres[0:10]

array([[ 0.,  1.],
       [ 1.,  2.],
       [ 2.,  3.],
       [ 3.,  5.],
       [ 4.,  6.],
       [ 5.,  8.],
       [ 6.,  9.],
       [ 7., 10.],
       [ 8., 11.],
       [ 9., 13.]])

In [ ]:
movie_corres[10,1]

array([ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10.])

In [39]:
# movie index in matrix user_prediction for movie id = 2355
iid = 2355.
iidx = np.where(movie_corres[:,1] == iid)
iidx

(array([1897]),)

In [43]:
iidx[0][0]

np.int64(1897)

In [44]:
custom_prediction[0, iidx]

array([[4.8746282]])

In [51]:
def get_custom_pred(user_id, movie_id, user_corres = user_corres, movie_corres = movie_corres, custom_pred_mat = custom_prediction):
    
    iidx = np.where(movie_corres[:,1] == movie_id)
    uidx = np.where(user_corres[:,1] == user_id)
    
    return custom_prediction[uidx, iidx][0][0]

In [52]:
get_custom_pred(1, 2355)

np.float64(4.874628201852936)

In [55]:
for i in range(10):
    print(f"UserID: {ratings_df.loc[i, "UserID"]}, MovieID: {ratings_df.loc[i, "MovieID"]}, true rating: {ratings_df.loc[i, "Rating"]}, custom collaborative prediction: {get_custom_pred(ratings_df.loc[i, "UserID"], ratings_df.loc[i, "MovieID"])} ")


UserID: 1, MovieID: 1193, true rating: 5, custom collaborative prediction: 4.821599140727186 
UserID: 1, MovieID: 661, true rating: 3, custom collaborative prediction: 4.921038170076707 
UserID: 1, MovieID: 914, true rating: 3, custom collaborative prediction: 4.89282334075204 
UserID: 1, MovieID: 3408, true rating: 4, custom collaborative prediction: 4.817163607340924 
UserID: 1, MovieID: 2355, true rating: 5, custom collaborative prediction: 4.874628201852936 
UserID: 1, MovieID: 1197, true rating: 3, custom collaborative prediction: 4.809683598258527 
UserID: 1, MovieID: 1287, true rating: 5, custom collaborative prediction: 4.907940477935489 
UserID: 1, MovieID: 2804, true rating: 5, custom collaborative prediction: 4.93846574579263 
UserID: 1, MovieID: 594, true rating: 4, custom collaborative prediction: 4.9670517715365055 
UserID: 1, MovieID: 919, true rating: 4, custom collaborative prediction: 4.96625482744612 


In [71]:
MAE = 0.0
RMSE = 0.0
df_ratio = 0.1
for i in range(int(df_ratio*len(ratings_df))):
    MAE += np.abs(ratings_df.loc[i, "Rating"]-get_custom_pred(ratings_df.loc[i, "UserID"], ratings_df.loc[i, "MovieID"]))
    RMSE += np.square(ratings_df.loc[i, "Rating"]-get_custom_pred(ratings_df.loc[i, "UserID"], ratings_df.loc[i, "MovieID"]))

MAE = MAE/(df_ratio*len(ratings_df))
RMSE = np.sqrt(RMSE/(df_ratio*len(ratings_df)))
print(f"MAE of custom collaborative RS: {MAE}, RMSE of custom collaborative RS: {RMSE}")


MAE of custom collaborative RS: 1.1165002892205693, RMSE of custom collaborative RS: 1.4289663178655059


In [86]:
for i in range(10):
    print(f"true rating: {ratings_df.loc[i, "Rating"]}") #,  surprise: {algo.predict(ratings_df.loc[i, "UserID"], ratings_df.loc[i, "MovieID"], verbose=True)[0]} ")


true rating: 5
true rating: 3
true rating: 3
true rating: 4
true rating: 5
true rating: 3
true rating: 5
true rating: 5
true rating: 4
true rating: 4


In [80]:
type(algo.predict(ratings_df.loc[i, "UserID"], ratings_df.loc[i, "MovieID"], verbose=True))

user: 1          item: 919        r_ui = None   est = 4.52   {'was_impossible': False}


surprise.prediction_algorithms.predictions.Prediction